# Human Action & Behavior Monitor — live GPU demo

**Press ▶ on the cell below (or Run All). That is the whole procedure.**

It sets everything up on this Kaggle T4 GPU (first time ≈ 4 min, then ≈ 1 min) and prints
a **client link**. Send that link to the client: it opens the live dashboard in any browser,
phone included, protected by a built-in access token.

In the dashboard the client can pick demo clips, **Upload** their own video, or press
**Go live** to use their own webcam — every frame is processed on this GPU.

Leave the cell running during the demo; it keeps the session alive and restarts the
server automatically if anything goes wrong. Press ■ (interrupt) when you are done.

*Needs (already set when created by `scripts/kaggle_auto.py`): Accelerator GPU T4 x2,
Internet On, and the `human-action-monitor` dataset as input.*

In [ ]:
# ▶ ONE CELL: setup + start + client link + keep-alive
import glob, json, os, pathlib, shutil, subprocess, sys, zipfile
from IPython.display import HTML, display

# Programs launched from here must not inherit Jupyter's inline chart backend.
os.environ["MPLBACKEND"] = "Agg"
APP = pathlib.Path("/kaggle/working/human-action-monitor")
if not (APP / "scripts" / "run_pipeline.py").exists():
    found = glob.glob("/kaggle/input/**/scripts/run_pipeline.py", recursive=True)
    if found:
        shutil.copytree(pathlib.Path(found[0]).parent.parent, APP)
    else:
        zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
        assert zips, "Add the 'human-action-monitor' dataset: right panel -> Add Input."
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall("/kaggle/working/_unzip")
        root = next(pathlib.Path("/kaggle/working/_unzip").rglob("run_pipeline.py")).parent.parent
        shutil.move(str(root), APP)
os.chdir(APP)

try:                                        # optional fixed token / ngrok via Add-ons -> Secrets
    from kaggle_secrets import UserSecretsClient
    for _k in ("ACCESS_TOKEN", "NGROK_AUTHTOKEN"):
        try: os.environ[_k] = UserSecretsClient().get_secret(_k)
        except Exception: pass
except ImportError:
    pass

def step(*args):
    r = get_ipython().system(f"{sys.executable} scripts/kaggle_launch.py " + " ".join(args))

state = pathlib.Path("/root/ham/state.json")
if not (state.exists() and json.loads(state.read_text()).get("setup_done")):
    step("setup")
step("start")

st = json.loads(state.read_text())
if st.get("public_url"):
    link = f"{st['public_url']}/?token={st['token']}"
    display(HTML(f'<div style="font-size:18px;padding:14px;border:2px solid #22c55e;'
                 f'border-radius:10px">Client link: <a href="{link}" target="_blank">'
                 f'<b>{link}</b></a></div>'))

if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
    step("status")                          # a background "Save Version" run: check and stop
else:
    step("monitor", "600")                  # keep alive up to 10 h; interrupt when done

### Optional — offline report
Processes any clip with the same pipeline and plays the annotated result here
(saved in `/kaggle/working/outputs`, downloadable from the Output tab).

In [ ]:
CLIP = "data/samples/demo-cooking.mp4"
!/root/ham/venv/bin/python scripts/process_video.py {CLIP} --out /kaggle/working/outputs
import json, pathlib
from IPython.display import Video, display
_s = json.load(open(f"/kaggle/working/outputs/{pathlib.Path(CLIP).stem}_summary.json"))
print(json.dumps(_s, indent=2))
if _s.get("annotated_video"):
    display(Video(_s["annotated_video"], embed=True, width=800))

In [ ]:
# Stop everything when the demo is over (GPU quota counts while it runs).
!python scripts/kaggle_launch.py stop